# env_functions

- This notebook is to sabe methods used along all the Munic Pipeline

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

In [0]:
def upsert_delta_table(df_source, table_name, merge_keys, ano_edicao, description):
    """Upsert a Spark DataFrame into a Delta table partitioned by ano_edicao.

    Automatically adds 'ano_edicao' to the source if not already present.
    Handles table creation, migration from non-partitioned tables, schema
    evolution via ALTER TABLE, and MERGE-based upsert with composite keys.

    Parameters
    ----------
    df_source    : Spark DataFrame to upsert
    table_name   : fully qualified target table name
    merge_keys   : list of column names for the MERGE condition
    ano_edicao   : survey edition year (int)
    description  : userMetadata string for the Delta operation
    """
    # Ensure ano_edicao exists in the source; place it second (after the entity key)
    source_cols = df_source.columns
    if "ano_edicao" not in source_cols:
        first_col = source_cols[0]
        rest_cols = source_cols[1:]
        df_source = df_source.withColumns({"ano_edicao": F.lit(ano_edicao)})
        df_source = df_source.select(first_col, "ano_edicao", *rest_cols)
        source_cols = [first_col, "ano_edicao", *rest_cols]

    # e.g, "target.ano_edicao = source.ano_edicao AND target.cod_munic = source.cod_munic"
    merge_condition = " AND ".join([f"target.{k} = source.{k}" for k in merge_keys])

    # Case 1: table doesn't exist yet - create it
    if not spark.catalog.tableExists(table_name):
        (df_source.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .partitionBy("ano_edicao")
            .option("userMetadata", description)
            .saveAsTable(table_name))
        print(f"Created: {table_name} - year {ano_edicao}")
        return

    deltaTable = DeltaTable.forName(spark, table_name)

    # Case 2: existing table without ano_edicao - migrate
    # Store the DataFrame once to avoid repeated Analyze RPCs on Spark Connect
    target_df = deltaTable.toDF()
    target_cols = target_df.columns
    if "ano_edicao" not in target_cols:
        print(f"Migrating {table_name}: adding ano_edicao partition column...")
        if "reference_year" in target_cols:
            # Backfill ano_edicao from the existing reference_year column
            existing_df = target_df.withColumns(
                {"ano_edicao": F.col("reference_year").cast("int")}
            )
        else:
            # No reference_year column (e.g. dictionary table) - use the parameter value
            existing_df = target_df.withColumns(
                {"ano_edicao": F.lit(ano_edicao)}
            )
        # Place ano_edicao second — after the entity key, not as trailing metadata
        first_col = target_cols[0]
        rest_cols = target_cols[1:]
        existing_df = existing_df.select(first_col, "ano_edicao", *rest_cols)
        (existing_df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .partitionBy("ano_edicao")
            .option("userMetadata", "Migration: added ano_edicao partition")
            .saveAsTable(table_name))
        deltaTable = DeltaTable.forName(spark, table_name)
        target_cols = [first_col, "ano_edicao", *rest_cols]
    else:
        # Ensure ano_edicao is second (after the entity key from merge_keys)
        entity_key = next(k for k in merge_keys if k != "ano_edicao")
        if len(target_cols) < 2 or target_cols[0] != entity_key or target_cols[1] != "ano_edicao":
            # Move ano_edicao first, then entity_key first (net: entity_key, ano_edicao, ...)
            spark.sql(f"ALTER TABLE {table_name} ALTER COLUMN ano_edicao FIRST")
            spark.sql(f"ALTER TABLE {table_name} ALTER COLUMN {entity_key} FIRST")
            deltaTable = DeltaTable.forName(spark, table_name)
            target_cols = [entity_key, "ano_edicao"] + [c for c in target_cols if c not in (entity_key, "ano_edicao")]

    # Schema evolution: add any new source columns missing in target
    missing_cols = [c for c in source_cols if c not in target_cols]
    if missing_cols:
        for col in missing_cols:
            spark.sql(f"ALTER TABLE {table_name} ADD COLUMN {col} STRING")
        print(f"Added columns to {table_name}: {missing_cols}")
        deltaTable = DeltaTable.forName(spark, table_name)

    # MERGE: update matched rows, insert new rows
    (deltaTable.alias("target")
        .merge(df_source.alias("source"), merge_condition)
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    print(f"MERGE done: {table_name} - year {ano_edicao}")


